# 07 — Model Tuning & Calibration

In this notebook, we perform hyperparameter tuning using chronological cross-validation (`TimeSeriesSplit`) on the best performing baseline classifiers. We then calibrate the final model's probabilities and serialize the final artifacts.

In [1]:
import pandas as pd
import numpy as np
import joblib
import json
from datetime import datetime
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, 
    log_loss, classification_report, confusion_matrix
)
from xgboost import XGBClassifier

## Load Datasets

In [2]:
train_df = pd.read_csv('../data/processed/train.csv')
test_df = pd.read_csv('../data/processed/test.csv')

feature_cols = [
    'HomeForm', 'AwayForm', 'HomeGoalsAvg5', 'AwayGoalsAvg5',
    'HomeGoalsConcededAvg5', 'AwayGoalsConcededAvg5', 'HomeWinRate5', 'AwayWinRate5',
    'HomeH2HForm5', 'HomePosition', 'AwayPosition', 'HomeGoalDiff5', 'AwayGoalDiff5',
    'HomeRestDays', 'AwayRestDays'
]

X_train = train_df[feature_cols]
y_train = train_df['FullTimeResult']
X_test = test_df[feature_cols]
y_test = test_df['FullTimeResult']

# Map classes to numeric for XGBoost
class_map = {'A': 0, 'D': 1, 'H': 2}
inv_class_map = {0: 'A', 1: 'D', 2: 'H'}
y_train_num = y_train.map(class_map)
y_test_num = y_test.map(class_map)

## Setup Chronological Validation (`TimeSeriesSplit`)

In [3]:
# We use a 5-fold TimeSeriesSplit to avoid random shuffling of time-series match data
tscv = TimeSeriesSplit(n_splits=5)
print(tscv)

TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None)


## Hyperparameter Tuning: Random Forest

In [4]:
# Define the parameter grid for Random Forest
rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 8, 12],
    'min_samples_split': [5, 10],
    'min_samples_leaf': [4, 8],
    'class_weight': ['balanced', None]
}

rf_clf = RandomForestClassifier(random_state=42)
rf_grid = GridSearchCV(estimator=rf_clf, param_grid=rf_param_grid, cv=tscv, scoring='neg_log_loss', n_jobs=-1)
rf_grid.fit(X_train, y_train)

print('Best RF Parameters:', rf_grid.best_params_)
best_rf = rf_grid.best_estimator_

Best RF Parameters: {'class_weight': None, 'max_depth': 5, 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 200}


## Hyperparameter Tuning: XGBoost

In [5]:
# Define the parameter grid for XGBoost
xgb_param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_clf = XGBClassifier(random_state=42, eval_metric='mlogloss')
xgb_grid = GridSearchCV(estimator=xgb_clf, param_grid=xgb_param_grid, cv=tscv, scoring='neg_log_loss', n_jobs=-1)
xgb_grid.fit(X_train, y_train_num)

print('Best XGBoost Parameters:', xgb_grid.best_params_)
best_xgb = xgb_grid.best_estimator_

Best XGBoost Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}


## Evaluate Tuned Models on Test Set

In [6]:
# Evaluated tuned RF
rf_tuned_pred = best_rf.predict(X_test)
rf_tuned_prob = best_rf.predict_proba(X_test)
print('=== Tuned Random Forest ===')
print(classification_report(y_test, rf_tuned_pred))
print('Log Loss:', log_loss(y_test, rf_tuned_prob))

# Evaluated tuned XGBoost
xgb_tuned_pred_num = best_xgb.predict(X_test)
xgb_tuned_pred = pd.Series(xgb_tuned_pred_num).map(inv_class_map)
xgb_tuned_prob = best_xgb.predict_proba(X_test)
print('=== Tuned XGBoost ===')
print(classification_report(y_test, xgb_tuned_pred))
print('Log Loss:', log_loss(y_test, xgb_tuned_prob))

=== Tuned Random Forest ===
              precision    recall  f1-score   support

           A       0.53      0.35      0.42       539
           D       0.00      0.00      0.00       374
           H       0.51      0.90      0.65       722

    accuracy                           0.51      1635
   macro avg       0.35      0.42      0.36      1635
weighted avg       0.40      0.51      0.43      1635

Log Loss: 1.0039790837656677
=== Tuned XGBoost ===
              precision    recall  f1-score   support

           A       0.54      0.42      0.47       539
           D       0.00      0.00      0.00       374
           H       0.52      0.87      0.65       722

    accuracy                           0.52      1635
   macro avg       0.35      0.43      0.37      1635
weighted avg       0.41      0.52      0.44      1635

Log Loss: 0.9960612058639526


/home/avrbt/Documents/Projects/Prem_League/venv/lib64/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/avrbt/Documents/Projects/Prem_League/venv/lib64/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/avrbt/Documents/Projects/Prem_League/venv/lib64/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavio

## Model Calibration

Let's choose the best tuned model (based on lower test Log Loss and balanced F1) and apply calibration using `CalibratedClassifierCV` to output reliable probabilities.

In [7]:
# XGBoost usually has lower log loss after tuning. Let's calibrate best_xgb using 'sigmoid' (Platt scaling) or 'isotonic'.
# Since we do time-series split, we can calibrate using cross-validation (prefit or tscv).
# Let's fit the CalibratedClassifierCV on the full training data.
calibrated_model = CalibratedClassifierCV(estimator=best_xgb, method='sigmoid', cv=tscv)
calibrated_model.fit(X_train, y_train_num)

cal_pred_num = calibrated_model.predict(X_test)
cal_pred = pd.Series(cal_pred_num).map(inv_class_map)
cal_prob = calibrated_model.predict_proba(X_test)

print('=== Calibrated XGBoost ===')
print(classification_report(y_test, cal_pred))
print('Log Loss (Before Calib):', log_loss(y_test, xgb_tuned_prob))
print('Log Loss (After Calib):', log_loss(y_test, cal_prob))

=== Calibrated XGBoost ===
              precision    recall  f1-score   support

           A       0.52      0.41      0.46       539
           D       0.00      0.00      0.00       374
           H       0.52      0.86      0.65       722

    accuracy                           0.52      1635
   macro avg       0.35      0.43      0.37      1635
weighted avg       0.40      0.52      0.44      1635

Log Loss (Before Calib): 0.9960612058639526
Log Loss (After Calib): 1.0000465917252623


/home/avrbt/Documents/Projects/Prem_League/venv/lib64/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/avrbt/Documents/Projects/Prem_League/venv/lib64/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/avrbt/Documents/Projects/Prem_League/venv/lib64/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavio

## Final Model Serialization

In [8]:
# Save the calibrated XGBoost model as the final model
joblib.dump(calibrated_model, '../models/final_model.pkl')
joblib.dump(feature_cols, '../models/feature_columns.pkl')

# Compute final metrics
acc = accuracy_score(y_test, cal_pred)
macro_prec, macro_rec, macro_f1, _ = precision_recall_fscore_support(y_test, cal_pred, average='macro')
loss = log_loss(y_test, cal_prob)

# Create metadata file
metadata = {
    'model_name': 'Calibrated XGBoost Classifier',
    'training_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'features_used': feature_cols,
    'training_rows': len(train_df),
    'test_rows': len(test_df),
    'accuracy': float(acc),
    'macro_f1': float(macro_f1),
    'log_loss': float(loss),
    'class_labels': ['A', 'D', 'H']
}

with open('../models/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=4)

print('Saved final model, features list, and metadata successfully!')

Saved final model, features list, and metadata successfully!


/home/avrbt/Documents/Projects/Prem_League/venv/lib64/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
